# 📦 Notebook 1: Project Setup, Dataset Pipeline & Annotation Schema

**Project:** Do Image Captioning Models Fail Like Humans?  
**Phase:** 1 — Generalization Study  

---

## 🗂️ Expected Folder Structure (your layout)

```
code_p1/
├── config.py                      ← Edit Section 1 here ONLY
├── annotations/
│   ├── captions_val2014.json
│   └── instances_val2014.json
├── fixations/
│   ├── train/   (.mat files)
│   └── val/     (.mat files)
├── images/
│   ├── train/   (.jpg)
│   └── val/     (.jpg)
├── maps/
│   ├── train/   (.png saliency)
│   └── val/     (.png saliency)
└── 01_setup_and_dataset.ipynb
```


In [ ]:
# CELL 1 — Install dependencies
import subprocess, sys
for pkg in ['pycocotools','Pillow','pandas','tqdm','numpy',
            'matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg])
print('✅ Dependencies installed.')

In [ ]:
# CELL 2 — Import config and validate all paths
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.')))
import config as cfg

path_status = cfg.validate_paths(verbose=True)

In [ ]:
print(cfg.MODELS_TO_RUN)

In [ ]:
# CELL 3 — Standard imports
import os, json, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED        = cfg.SEED
SPLIT       = cfg.SPLIT
N_IMAGES    = cfg.N_IMAGES
SPATIAL_RES = cfg.SPATIAL_RES
OUTPUT_DIR  = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR

random.seed(SEED)
np.random.seed(SEED)
print(f'✅ split={SPLIT}, N={N_IMAGES}, res={SPATIAL_RES}x{SPATIAL_RES}')

## 📖 Section 1: Load COCO Annotations

In [ ]:
# CELL 4 — Load captions_val2014.json
print(f'Loading: {cfg.CAPTIONS_JSON}')
if not cfg.CAPTIONS_JSON.exists():
    raise FileNotFoundError(f'❌ Not found: {cfg.CAPTIONS_JSON.resolve()}')

with open(cfg.CAPTIONS_JSON) as f:
    coco_data = json.load(f)

captions_by_image = {}
for ann in coco_data['annotations']:
    captions_by_image.setdefault(ann['image_id'], []).append(ann['caption'])

id_to_file = {img['id']: img['file_name'] for img in coco_data['images']}

print(f'✅ {len(id_to_file)} images, {len(coco_data["annotations"])} captions')
sid = list(id_to_file.keys())[0]
print(f'   Sample id={sid}: {id_to_file[sid]}')
for c in captions_by_image.get(sid, [])[:2]:
    print(f'   → {c}')

In [ ]:
# CELL 5 — Find image_ids that have physical files on disk
images_dir = cfg.IMAGES_DIR
available_files = {f.name for f in images_dir.glob('*.jpg')}
print(f'images/{SPLIT}/: {len(available_files)} .jpg files')

valid_ids = [iid for iid, fname in id_to_file.items() if fname in available_files]
print(f'Matched annotation IDs: {len(valid_ids)}')

if len(valid_ids) == 0:
    print('⚠️  No matches — using all annotation IDs (images loaded from URL if missing)')
    valid_ids = list(id_to_file.keys())

In [ ]:
# CELL 6 — Cross-reference with maps/ and fixations/
maps_dir = cfg.MAPS_DIR
fix_dir  = cfg.FIXATIONS_DIR

available_maps = {f.stem for f in maps_dir.glob('*.png')} if maps_dir.exists() else set()
available_mats = {f.stem for f in fix_dir.glob('*.mat')}  if fix_dir.exists()  else set()

print(f'maps/{SPLIT}/:      {len(available_maps)} PNGs')
print(f'fixations/{SPLIT}/: {len(available_mats)} .mat files')

def to_stem(iid): return f'COCO_{SPLIT}2014_{iid:012d}'

ids_with_mats = [iid for iid in valid_ids if to_stem(iid) in available_mats]
ids_with_maps = [iid for iid in valid_ids if to_stem(iid) in available_maps]
USABLE_IDS    = list(set(ids_with_mats) | set(ids_with_maps)) or valid_ids

print(f'IDs with .mat: {len(ids_with_mats)}')
print(f'IDs with .png: {len(ids_with_maps)}')
print(f'Usable total:  {len(USABLE_IDS)}')

if len(USABLE_IDS) < N_IMAGES:
    print(f'⚠️  Only {len(USABLE_IDS)} usable — adjusting N_IMAGES')
    N_IMAGES = len(USABLE_IDS)

In [ ]:
# CELL 7 — Inspect .mat file structure (so you know what fields you have)
import scipy.io as sio

mat_files = list(cfg.FIXATIONS_DIR.glob('*.mat')) if cfg.FIXATIONS_DIR.exists() else []
if mat_files:
    mat = sio.loadmat(str(mat_files[0]))
    keys = [k for k in mat if not k.startswith('_')]
    print(f'Sample .mat: {mat_files[0].name}')
    print(f'Keys: {keys}')
    for k in keys:
        v = mat[k]
        print(f'  [{k}] type={type(v).__name__} ', end='')
        if hasattr(v,'shape'):
            print(f'shape={v.shape} dtype={v.dtype}', end='')
            if v.dtype.names: print(f' sub={v.dtype.names}', end='')
        print()
else:
    print(f'No .mat files in {cfg.FIXATIONS_DIR}')

In [ ]:
# CELL 8 — Test: load one image + saliency side-by-side
test_id    = USABLE_IDS[0]
test_fname = id_to_file[test_id]
img_path   = cfg.get_image_path(test_id, SPLIT, test_fname)
sal        = cfg.load_saliency(test_id, SPLIT, SPATIAL_RES, prefer='png')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
if img_path.exists():
    axes[0].imshow(Image.open(img_path).convert('RGB'))
else:
    axes[0].text(0.5, 0.5, f'Image not found\n{img_path}',
                 ha='center', va='center', transform=axes[0].transAxes)
axes[0].set_title(f'id={test_id}\n{test_fname}', fontsize=8)
axes[0].axis('off')

im = axes[1].imshow(sal.reshape(SPATIAL_RES, SPATIAL_RES), cmap='hot')
axes[1].set_title(f'Saliency ({SPATIAL_RES}×{SPATIAL_RES})\nsum={sal.sum():.4f}', fontsize=8)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.savefig(FIGURES_DIR/'test_image_saliency.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Image: {img_path.exists()} | Saliency: sum={sal.sum():.4f}')

## 📊 Section 2: Build Scaffold & Cache Saliency

In [ ]:
# CELL 9 — Sampling plan
SCENARIO_FRACTIONS = {
    'gender_ambiguity':0.20,'object_confusion':0.20,'context_mismatch':0.20,
    'counting_errors':0.15,'attribute_errors':0.15,'general_baseline':0.10
}
cat_counts = {c: int(f*N_IMAGES) for c,f in SCENARIO_FRACTIONS.items()}
cat_counts['general_baseline'] += N_IMAGES - sum(cat_counts.values())
for c,n in cat_counts.items(): print(f'  {c:<25} {n:3d}')

In [ ]:
# CELL 10 — Build annotation scaffold DataFrame
random.seed(SEED); np.random.seed(SEED)
sampled_ids = random.sample(USABLE_IDS, min(N_IMAGES, len(USABLE_IDS)))
cats = []; [cats.extend([c]*n) for c,n in cat_counts.items()]
random.shuffle(cats); cats = cats[:len(sampled_ids)]

records = []
for i, iid in enumerate(sampled_ids):
    gt_caps = captions_by_image.get(iid, ['N/A'])
    fname   = id_to_file.get(iid, f'COCO_{SPLIT}2014_{iid:012d}.jpg')
    stem    = to_stem(iid)
    records.append({
        'image_id':iid,'file_name':fname,'split':SPLIT,
        'scenario_category':cats[i],
        'has_saliency_png': stem in available_maps,
        'has_fixation_mat': stem in available_mats,
        'ground_truth_caps':str(gt_caps),
        'primary_gt_caption':gt_caps[0],
        'n_gt_captions':len(gt_caps),
        'blip_caption':None,'blip2_caption':None,
        'ofa_caption':None,'vit_gpt2_caption':None,
        'bleu4_blip':None,'bleu4_blip2':None,'bleu4_ofa':None,'bleu4_vit_gpt2':None,
        'cider_blip':None,'meteor_blip':None,'rougeL_blip':None,
        'human_correct_blip':None,'human_correct_blip2':None,
        'human_correct_ofa':None,'human_correct_vit_gpt2':None,
        'error_type_blip':None,'error_type_blip2':None,
        'error_type_ofa':None,'error_type_vit_gpt2':None,
        'error_token_blip':None,'error_token_blip2':None,
        'error_token_ofa':None,'error_token_vit_gpt2':None,
        'js_div_mean_blip':None,'kl_div_mean_blip':None,
        'error_token_js_blip':None,'other_tokens_js_blip':None,
    })

df = pd.DataFrame(records)
print(f'✅ Scaffold: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'   .mat available: {df["has_fixation_mat"].sum()} | .png: {df["has_saliency_png"].sum()}')
df.head(3)

In [ ]:
# CELL 11 — Precompute and cache all saliency maps
print(f'🔄 Caching saliency for {len(df)} images (.mat preferred)...')
saliency_cache = {}
src_log = {'mat':0,'png':0,'sim':0}

for _, row in tqdm(df.iterrows(), total=len(df)):
    iid = row['image_id']
    if row['has_fixation_mat']:
        sal = cfg.load_saliency_from_mat(iid, SPLIT, SPATIAL_RES)
        src_log['mat'] += 1
    elif row['has_saliency_png']:
        sal = cfg.load_saliency_from_png(iid, SPLIT, SPATIAL_RES)
        src_log['png'] += 1
    else:
        s = cfg._simulate_saliency(SPATIAL_RES, iid).flatten()
        sal = (s+1e-10)/(s+1e-10).sum()
        src_log['sim'] += 1
    saliency_cache[iid] = sal

np.save(OUTPUT_DIR/'saliency_cache.npy', saliency_cache)
print(f'✅ Cached: mat={src_log["mat"]} png={src_log["png"]} sim={src_log["sim"]}')
print(f'💾 saliency_cache.npy saved')

In [ ]:
# CELL 12 — Visualize 6 image+saliency pairs
n_show = min(6, len(df))
fig, axes = plt.subplots(2, n_show, figsize=(2.5*n_show, 5))
fig.suptitle('Sample Images + Saliency Maps', fontsize=12, fontweight='bold')

for ci, (_, row) in enumerate(df.head(n_show).iterrows()):
    iid = row['image_id']
    ip  = cfg.get_image_path(iid, SPLIT, row['file_name'])
    if ip.exists():
        axes[0,ci].imshow(Image.open(ip).convert('RGB'))
    else:
        axes[0,ci].text(0.5,0.5,'N/A',ha='center',va='center',
                         transform=axes[0,ci].transAxes, fontsize=7)
    axes[0,ci].set_title(f'id={iid}',fontsize=7); axes[0,ci].axis('off')

    sal = saliency_cache[iid].reshape(SPATIAL_RES, SPATIAL_RES)
    src = 'mat' if row['has_fixation_mat'] else ('png' if row['has_saliency_png'] else 'sim')
    axes[1,ci].imshow(sal, cmap='hot')
    axes[1,ci].set_title(f'[{src}]', fontsize=7); axes[1,ci].axis('off')

plt.tight_layout()
fig.savefig(FIGURES_DIR/'sample_saliency_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 13 — Save scaffold + metadata
df.to_csv(OUTPUT_DIR/'annotation_scaffold.csv', index=False)
meta = {
    'split':SPLIT,'n_images':len(df),'seed':SEED,'spatial_res':SPATIAL_RES,
    'models':cfg.MODELS_TO_RUN,'saliency_sources':src_log,
    'scenario_counts':df['scenario_category'].value_counts().to_dict(),
    'scaffold_hash': hashlib.md5(
        df[['image_id','scenario_category']].to_json().encode()
    ).hexdigest()
}
with open(OUTPUT_DIR/'metadata.json','w') as f:
    json.dump(meta, f, indent=2)

print('💾 annotation_scaffold.csv saved')
print('💾 metadata.json saved')
print('\n✅ Notebook 1 COMPLETE — run 02_caption_generation.ipynb next')